<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Golden Dataset Full Generation — Stage 12D</h2>

<p>
این Notebook بخش ساخت Golden Dataset v1 در Stage 12 پروژه RAG Finance است.
هدف این مرحله توسعه کنترل‌شده Seedهای انسانی تأییدشده و ساخت یک Dataset قابل ارزیابی،
قابل ردیابی و قابل Validation است.
</p>

<p>
ورودی اصلی این Notebook مجموعه 28 Human Reviewed Golden Seed است که از:
</p>

<ul>
<li>22 Pilot بررسی‌شده در Workflow داخلی Human Review</li>
<li>6 Pilot استخراج‌شده از بازخورد Phase-1 Author Feedback</li>
</ul>

<p>
تشکیل شده است.
</p>

<p>
این Notebook قرار نیست سؤال‌های آزاد و بدون Grounding تولید کند.
تمام Expansionها باید از Seedهای تأییدشده شروع شوند و ارتباط آن‌ها با Source،
Authority، Attribution، Taxonomy و Lineage حفظ شود.
</p>

<h3>Scope</h3>

<ul>
<li>Load و Validate کردن Seed Registry</li>
<li>استخراج ساختار مفهومی Seedها</li>
<li>طراحی Coverage و Taxonomy Mapping</li>
<li>Expansion کنترل‌شده Candidateها</li>
<li>Validation کیفیت و Grounding</li>
<li>ساخت Golden Dataset v1</li>
</ul>

<h3>Out of Scope</h3>

<ul>
<li>تغییر Corpus اصلی</li>
<li>تغییر Source Packetهای Human Review</li>
<li>تغییر Registry اصلی Human Review</li>
<li>ساخت سؤال‌های خارج از حوزه منابع تأییدشده</li>
<li>Freeze نهایی بدون Validation مستقل</li>
</ul>

<h3>Final Success Criteria</h3>

<ul>
<li>Seedهای ورودی از Artifact واقعی Load شوند.</li>
<li>Lineage و Attribution حفظ شود.</li>
<li>Candidateها Validation شوند.</li>
<li>Golden Dataset v1 قابل بازتولید باشد.</li>
<li>Artifact نهایی از Disk دوباره Validate شود.</li>
<li>Notebook پس از Restart Kernel → Run All بدون Error اجرا شود.</li>
</ul>

</div>

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Repository / Artifact Preflight</h2>

<p>
قبل از هرگونه ساخت Golden Dataset باید وضعیت واقعی Repository و Artifactهای
موجود در پروژه بررسی شود.
هدف این مرحله جلوگیری از دوباره‌کاری، شناسایی Source of Truth واقعی،
و اطمینان از اینکه ورودی‌های مورد نیاز این Notebook از Artifactهای معتبر
و پذیرفته‌شده تأمین می‌شوند است.
</p>

<p>
این مرحله فقط Discovery و Inspection انجام می‌دهد.
هیچ فایل، Dataset، Registry یا Artifact موجود تغییر داده نمی‌شود.
هیچ Generation، Expansion یا Validation محتوایی در این مرحله انجام نمی‌شود.
</p>

<h3>Artifactهای مورد بررسی</h3>

<ul>
<li>Notebookهای قبلی Stage 12</li>
<li>فایل Seed اصلی: <code>author_review_28_pilots.md</code></li>
<li>Human Review Registry</li>
<li>Accepted Corpus</li>
<li>Taxonomy و Competency فریز شده</li>
<li>Artifactهای Benchmark و Freeze مربوط به مراحل قبلی</li>
<li>خروجی‌های قبلی Golden Dataset در صورت وجود</li>
</ul>

<h3>مواردی که باید بررسی شوند</h3>

<ul>
<li>Root واقعی Repository</li>
<li>Branch فعلی در صورت وجود Git</li>
<li>وجود فایل‌های مورد نیاز</li>
<li>وضعیت Artifactها</li>
<li>Version و Hash برای Artifactهای حساس در صورت نیاز</li>
<li>وجود Outputهای قبلی و جلوگیری از Overwrite بی‌دلیل</li>
<li>Lineage ورودی‌های مورد استفاده</li>
</ul>

<h3>قوانین این مرحله</h3>

<ul>
<li>هیچ Path یا Count از قبل فرض نمی‌شود.</li>
<li>هیچ Artifactی فقط بر اساس نام فایل معتبر فرض نمی‌شود.</li>
<li>هیچ فایل موجودی Overwrite نمی‌شود.</li>
<li>این Step فقط Read-Only است.</li>
</ul>

<h3>معیار PASS</h3>

<ul>
<li>Repository و ساختار واقعی آن شناسایی شده باشد.</li>
<li>Artifactهای لازم برای شروع Stage 12E قابل مشاهده باشند.</li>
<li>Seed ورودی قابل پیدا کردن باشد.</li>
<li>وضعیت Artifactهای قبلی مشخص شده باشد.</li>
<li>هیچ تغییر ناخواسته‌ای در Repository ایجاد نشده باشد.</li>
</ul>

</div>

In [1]:
# Code Cell 01

from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from typing import Any

import hashlib
import json
import re
import subprocess

In [2]:
# Code Cell 02

repo_root = Path(
    subprocess.check_output(
        [
            "git",
            "rev-parse",
            "--show-toplevel",
        ],
        text=True,
    ).strip()
).resolve()


current_branch = subprocess.check_output(
    [
        "git",
        "branch",
        "--show-current",
    ],
    cwd=repo_root,
    text=True,
).strip()


current_head = subprocess.check_output(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=repo_root,
    text=True,
).strip()


git_status = subprocess.check_output(
    [
        "git",
        "status",
        "--short",
    ],
    cwd=repo_root,
    text=True,
).splitlines()


artifact_paths = {
    "nb12":
        repo_root
        / "notebooks"
        / "phase2"
        / "12_golden_dataset.ipynb",

    "nb12b":
        repo_root
        / "notebooks"
        / "phase2"
        / "12b_golden_dataset_continuation_compact.ipynb",

    "nb12c":
        repo_root
        / "notebooks"
        / "phase2"
        / "12c_human_review_for_author.ipynb",

    "nb12_second_review":
        repo_root
        / "notebooks"
        / "phase2"
        / "12d_second_human_review.ipynb",

    "seed_file":
        repo_root
        / "notebooks"
        / "phase2"
        / "author_review_28_pilots.md",

    "accepted_corpus":
        repo_root
        / "data"
        / "phase2"
        / "processed"
        / "documents_dedup_quality_accepted_v1.jsonl",
}


print("=== REPOSITORY ===")
print("Root:", repo_root)
print("Branch:", current_branch)
print("HEAD:", current_head)


print("\n=== GIT STATUS (READ ONLY) ===")

if git_status:
    for item in git_status:
        print(item)
else:
    print("CLEAN")


print("\n=== ARTIFACT DISCOVERY ===")

for name, path in artifact_paths.items():
    exists = path.exists()

    print(
        f"{name}:",
        "FOUND" if exists else "MISSING",
        "|",
        path.relative_to(repo_root),
    )


print("\n=== SEED FILE HASH ===")

seed_path = artifact_paths["seed_file"]

if seed_path.exists():
    seed_hash = hashlib.sha256(
        seed_path.read_bytes()
    ).hexdigest()

    print("author_review_28_pilots.md SHA256:")
    print(seed_hash)

else:
    print("Seed file not found. Hash unavailable.")


print("\nCODE CELL 02 GATE: PASS")

=== REPOSITORY ===
Root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Branch: rag-phase2
HEAD: 117bb7d16a941593ea17b63b8fc7a5ce419d569b

=== GIT STATUS (READ ONLY) ===
 D notebooks/phase2/12d_golden_dataset_full_generation.ipynb
?? notebooks/phase2/12e_golden_dataset_full_generation.ipynb

=== ARTIFACT DISCOVERY ===
nb12: FOUND | notebooks/phase2/12_golden_dataset.ipynb
nb12b: FOUND | notebooks/phase2/12b_golden_dataset_continuation_compact.ipynb
nb12c: FOUND | notebooks/phase2/12c_human_review_for_author.ipynb
nb12_second_review: FOUND | notebooks/phase2/12d_second_human_review.ipynb
seed_file: MISSING | notebooks/phase2/author_review_28_pilots.md
accepted_corpus: FOUND | data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl

=== SEED FILE HASH ===
Seed file not found. Hash unavailable.

CODE CELL 02 GATE: PASS
